In [1]:
import pandas as pd
import numpy as np
import re

In [3]:
csv_path = "C:/Users/14794/Desktop/Longterm memory/hurst_CI_field_bootstrap_US_90.csv"  # <- change if needed

df = pd.read_csv(csv_path)


df["H_label"] = df["series"].str.extract(r"^(H\d)\b", expand=False)

# Keep only the 3 series we care about
df = df[df["H_label"].isin(["H1", "H2", "H3"])].copy()

# Desired order and labels for the table
order = ["H1", "H2", "H3"]
pretty = {
    "H1": "H1 (Sₜ)",
    "H2": "H2 (Vₜ)",
    "H3": "H3 (Rₜ)",
}

def make_sum_ci_table_90(period_df: pd.DataFrame, decimals: int = 4) -> pd.DataFrame:
    """
    Build a 3x3 table where entry (i,j) is the CI interval for H_i + H_j:
        [CI_lo(i)+CI_lo(j), CI_hi(i)+CI_hi(j)]
    Diagonal entries correspond to 2*H_i: [2*CI_lo(i), 2*CI_hi(i)].
    """
    # Ensure we have exactly one row per H1/H2/H3
    period_df = period_df.drop_duplicates(subset=["H_label"]).set_index("H_label").loc[order]
    if period_df.index.isnull().any() or len(period_df) != 3:
        raise ValueError("Each period must contain exactly one row for each of H1, H2, H3.")

    lo = period_df["CI_90_lo"].astype(float).to_numpy()
    hi = period_df["CI_90_hi"].astype(float).to_numpy()

    # Pairwise sums via broadcasting
    lo_sum = lo[:, None] + lo[None, :]
    hi_sum = hi[:, None] + hi[None, :]

    # Format as "[lower, upper]"
    fmt = lambda x: f"{x:.{decimals}f}"
    out = np.empty((3, 3), dtype=object)
    for i in range(3):
        for j in range(3):
            out[i, j] = f"[{fmt(lo_sum[i, j])}, {fmt(hi_sum[i, j])}]"

    table = pd.DataFrame(out, index=[pretty[h] for h in order], columns=[pretty[h] for h in order])
    return table

# Build tables for each period
tables = {}
for period, g in df.groupby("period", sort=False):
    tables[period] = make_sum_ci_table_90(g, decimals=4)

# Print tables
for period, tbl in tables.items():
    print("\n" + "=" * 80)
    print(period)
    print(tbl.to_string())


Period 1
                  H1 (Sₜ)           H2 (Vₜ)           H3 (Rₜ)
H1 (Sₜ)  [0.8936, 1.2658]  [0.9239, 1.2886]  [0.7372, 1.1096]
H2 (Vₜ)  [0.9239, 1.2886]  [0.9543, 1.3114]  [0.7675, 1.1324]
H3 (Rₜ)  [0.7372, 1.1096]  [0.7675, 1.1324]  [0.5808, 0.9535]

Period 2
                  H1 (Sₜ)           H2 (Vₜ)           H3 (Rₜ)
H1 (Sₜ)  [0.8263, 1.0226]  [0.8321, 1.0263]  [0.8906, 1.0824]
H2 (Vₜ)  [0.8321, 1.0263]  [0.8380, 1.0301]  [0.8964, 1.0862]
H3 (Rₜ)  [0.8906, 1.0824]  [0.8964, 1.0862]  [0.9549, 1.1423]

Period 3
                  H1 (Sₜ)           H2 (Vₜ)           H3 (Rₜ)
H1 (Sₜ)  [0.6818, 0.9684]  [0.6759, 0.9597]  [0.7347, 1.0242]
H2 (Vₜ)  [0.6759, 0.9597]  [0.6700, 0.9509]  [0.7288, 1.0154]
H3 (Rₜ)  [0.7347, 1.0242]  [0.7288, 1.0154]  [0.7876, 1.0799]

Period 4
                  H1 (Sₜ)           H2 (Vₜ)           H3 (Rₜ)
H1 (Sₜ)  [0.8110, 1.2873]  [0.7149, 1.1766]  [0.7305, 1.1771]
H2 (Vₜ)  [0.7149, 1.1766]  [0.6189, 1.0660]  [0.6345, 1.0665]
H3 (Rₜ)  [0.7305, 1.1771]  [0.